# Fleet Spend Control for LangGraph Multi-Agent Systems

[![PyPI](https://img.shields.io/pypi/v/figuard.svg)](https://pypi.org/project/figuard/)
[![GitHub](https://img.shields.io/badge/github-figuard%2Ffiguard--core-blue)](https://github.com/figuard/figuard-core)

This cookbook shows how to use [FiGuard](https://github.com/figuard/figuard-core) to enforce per-agent spend caps in a LangGraph multi-agent system.

**The problem:** LangGraph makes it easy to build supervisor graphs where one orchestrator delegates to specialist sub-agents. Without a control layer, a runaway sub-agent can consume the entire shared budget — starving the rest of the fleet.

**The solution:** FiGuard's fleet budgets. The orchestrator creates a shared ceiling ($1,000), then issues a **delegation token** to each sub-agent with a hard per-agent cap. Each sub-agent spends only against its own token — it can never exceed its cap, and every dollar it spends reduces the shared ceiling too.

```
Orchestrator — fleet budget $1,000
  ├── flight-booker  ←  delegation token, cap $600
  └── hotel-booker   ←  delegation token, cap $500
```

**No account required** — all examples connect to a shared public sandbox automatically.

## What this notebook covers

1. Fleet budget setup and delegation tokens
2. Sub-agent booking loop — authorized and denied cases
3. Wiring delegation tokens into a real LangGraph graph
4. Spend tree and ledger audit
5. `void_tree` — atomically cancel a graph run and release all reservations

## Setup

In [ ]:
%pip install -qU "figuard[langchain]" langgraph

FiGuard connects to a shared public sandbox by default — no API key or account needed. For production, self-host FiGuard and set `FIGUARD_API_KEY` / `FIGUARD_BASE_URL`.

## Part 1: Fleet budget and delegation tokens

The orchestrator owns the fleet budget and issues scoped delegation tokens to each sub-agent. Sub-agents never see the fleet session token — they only get their own capped token.

FiGuard enforces two limits on every sub-agent authorization:
- **Per-agent cap** (`DELEGATE_CAP_EXCEEDED`) — the agent cannot exceed its own token's limit
- **Fleet ceiling** (`BUDGET_EXHAUSTED`) — the sum of all sub-agent spend cannot exceed the fleet total

Whichever limit is hit first wins.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient()  # zero-config: connects to shared sandbox

# Orchestrator creates the fleet budget — shared $1,000 ceiling
fleet = client.create_budget(
    user_id="travel_orchestrator",
    total_limit=1000.00,
    currency="USD",
    expires_in="24h",
)
print(f"Fleet budget: {fleet.id}  available: ${fleet.available_quantity:.2f}")

# Issue one delegation token per sub-agent.
# The orchestrator authenticates with the fleet session token — sub-agents never see it.
fleet_token = fleet.primary_token.session_token

flight_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_token,  # orchestrator authenticates with fleet token
    label="flight-booker",
    caps=[{"category": "flights", "limit": 600.00}],
    expires_in="24h",
)

hotel_token = client.create_delegation_token(
    budget_id=fleet.id,
    session_token=fleet_token,  # orchestrator authenticates with fleet token
    label="hotel-booker",
    caps=[{"category": "hotels", "limit": 500.00}],
    expires_in="24h",
)

print(f"Flight-booker token: {flight_token.session_token[:20]}...  cap: $600")
print(f"Hotel-booker token:  {hotel_token.session_token[:20]}...  cap: $500")


## Part 2: Sub-agent booking loop

This example drives `FiGuardCallbackHandler` directly — no LLM required — to show exactly what happens when sub-agents try to spend.

Scenario:
- `flight-booker` books two flights ($280 + $290 = $570 total) — both authorized, within its $600 cap
- `hotel-booker` tries a $550 hotel — **denied** (`DELEGATE_CAP_EXCEEDED`, cap is $500)
- `hotel-booker` tries a cheaper $420 hotel — authorized

Even though the fleet has headroom when the $550 hotel is attempted, the per-agent cap blocks it.

In [ ]:
from uuid import uuid4
from langchain_core.tools import ToolException
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler

# Each sub-agent gets its own handler wired to its delegation token
flight_handler = FiGuardCallbackHandler(
    client=client,
    session_token=flight_token.session_token,
    agent_id="flight-booker",
    tool_category_map={"book_flight": "flights"},
)

hotel_handler = FiGuardCallbackHandler(
    client=client,
    session_token=hotel_token.session_token,
    agent_id="hotel-booker",
    tool_category_map={"book_hotel": "hotels"},
)

def run_tool(handler, tool_name, amount, description):
    """Simulates a tool call through FiGuardCallbackHandler."""
    run_id = uuid4()
    try:
        handler.on_tool_start(
            {"name": tool_name},
            f'{{"amount": {amount}, "description": "{description}"}}',
            run_id=run_id,
        )
        handler.on_tool_end(f"{description} confirmed", run_id=run_id)
        return True, None
    except ToolException as e:
        return False, str(e)

print("--- flight-booker (cap: $600) ---")
for amount, desc in [(280.00, "NYC to LAX"), (290.00, "LAX to NYC return")]:
    ok, err = run_tool(flight_handler, "book_flight", amount, desc)
    if ok:
        print(f"  ✓  book_flight    ${amount:>7.2f}   AUTHORIZED  ({desc})")
    else:
        print(f"  ✗  book_flight    ${amount:>7.2f}   {err}")

print()
print("--- hotel-booker (cap: $500) ---")
for amount, desc in [(550.00, "Grand Hyatt 3 nights"), (420.00, "Holiday Inn 3 nights")]:
    ok, err = run_tool(hotel_handler, "book_hotel", amount, desc)
    if ok:
        print(f"  ✓  book_hotel     ${amount:>7.2f}   AUTHORIZED  ({desc})")
    else:
        print(f"  ✗  book_hotel     ${amount:>7.2f}   {err}")

# Fleet summary
updated = client.get_budget(fleet.id)
print(f"\nFleet:  spent=${updated.quantity_spent:.2f}  reserved=${updated.quantity_reserved:.2f}  available=${updated.available_quantity:.2f}")

Expected output:
```
--- flight-booker (cap: $600) ---
  ✓  book_flight      $280.00   AUTHORIZED  (NYC to LAX)
  ✓  book_flight      $290.00   AUTHORIZED  (LAX to NYC return)

--- hotel-booker (cap: $500) ---
  ✗  book_hotel       $550.00   FiGuard DENIED: DELEGATE_CAP_EXCEEDED — $500.00 cap, $550.00 requested
  ✓  book_hotel       $420.00   AUTHORIZED  (Holiday Inn 3 nights)

Fleet:  spent=$990.00  reserved=$0.00  available=$10.00
```

The $550 hotel was denied even though the fleet still had headroom — the hotel-booker's $500 cap was the binding constraint. The LLM (when running in a real graph) receives the denial string `"FiGuard DENIED: DELEGATE_CAP_EXCEEDED — $500.00 cap, $550.00 requested"` as the tool result and can adapt: suggest a cheaper hotel, ask the user to increase the budget, or stop.

## Part 3: Wiring into a real LangGraph graph

In a real multi-agent LangGraph graph, each sub-agent node gets its own `FiGuardCallbackHandler` passed through `RunnableConfig`. The orchestrator node creates the fleet budget, issues delegation tokens, and hands each node its token.

Key points:
- Each sub-agent node calls `.invoke(state, config={"callbacks": [its_handler]})` — not the fleet handler
- The orchestrator never passes the fleet session token down
- `handle_tool_error=True` on any inner `AgentExecutor` ensures denials reach the LLM as tool results

The code below shows the structure without requiring an OpenAI API key — replace `...` comments with your model calls.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from figuard import FiGuardClient
from figuard.integrations.langchain import FiGuardCallbackHandler

# ── State ──────────────────────────────────────────────────────────────────────

class TripState(TypedDict):
    messages: Annotated[list, lambda x, y: x + y]
    # Delegation tokens are stored in state so each node can retrieve its own
    flight_session_token: str
    hotel_session_token: str
    # Track the fleet budget ID for the spend tree at the end
    fleet_budget_id: str

# ── FiGuard setup (orchestrator node) ─────────────────────────────────────────

def orchestrator_node(state: TripState) -> TripState:
    """Creates the fleet budget and issues delegation tokens to sub-agents."""
    client = FiGuardClient()

    fleet = client.create_budget(
        user_id="travel_orchestrator",
        total_limit=1000.00,
        currency="USD",
        expires_in="24h",
    )

    fleet_token = fleet.primary_token.session_token

    flight_token = client.create_delegation_token(
        budget_id=fleet.id,
        session_token=fleet_token,
        label="flight-booker",
        caps=[{"category": "flights", "limit": 600.00}],
    )
    hotel_token = client.create_delegation_token(
        budget_id=fleet.id,
        session_token=fleet_token,
        label="hotel-booker",
        caps=[{"category": "hotels", "limit": 500.00}],
    )

    # Store tokens in state — sub-agent nodes will read them
    return {
        **state,
        "flight_session_token": flight_token.session_token,
        "hotel_session_token": hotel_token.session_token,
        "fleet_budget_id": fleet.id,
        "messages": [("system", f"Fleet budget created: ${fleet.total_limit:.0f}")],
    }

# ── Sub-agent nodes ────────────────────────────────────────────────────────────

def flight_booker_node(state: TripState, config=None) -> TripState:
    """Specialist agent for booking flights.
    In a real graph, this calls an LLM with flight-booking tools.
    The FiGuardCallbackHandler is passed via config["callbacks"].
    """
    client = FiGuardClient()
    handler = FiGuardCallbackHandler(
        client=client,
        session_token=state["flight_session_token"],  # delegation token, not fleet token
        agent_id="flight-booker",
        tool_category_map={"book_flight": "flights"},
    )

    # Real usage:
    # result = flight_executor.invoke(
    #     {"messages": state["messages"]},
    #     config={"callbacks": [handler]},
    # )

    # Placeholder — return stub message
    return {**state, "messages": [("assistant", "[flight-booker] Flight booked within $600 cap.")]}


def hotel_booker_node(state: TripState, config=None) -> TripState:
    """Specialist agent for booking hotels."""
    client = FiGuardClient()
    handler = FiGuardCallbackHandler(
        client=client,
        session_token=state["hotel_session_token"],   # delegation token, not fleet token
        agent_id="hotel-booker",
        tool_category_map={"book_hotel": "hotels"},
    )

    # Real usage:
    # result = hotel_executor.invoke(
    #     {"messages": state["messages"]},
    #     config={"callbacks": [handler]},
    # )

    return {**state, "messages": [("assistant", "[hotel-booker] Hotel booked within $500 cap.")]}

# ── Build the graph ────────────────────────────────────────────────────────────

graph = StateGraph(TripState)
graph.add_node("orchestrator", orchestrator_node)
graph.add_node("flight_booker", flight_booker_node)
graph.add_node("hotel_booker", hotel_booker_node)

graph.set_entry_point("orchestrator")
graph.add_edge("orchestrator", "flight_booker")
graph.add_edge("orchestrator", "hotel_booker")  # runs in parallel
graph.add_edge("flight_booker", END)
graph.add_edge("hotel_booker", END)

app = graph.compile()

# Run
result = app.invoke({
    "messages": [("human", "Book a round-trip flight and hotel for a 3-day trip to Los Angeles.")],
    "flight_session_token": "",
    "hotel_session_token": "",
    "fleet_budget_id": "",
})

print("Graph complete. Fleet budget ID:", result["fleet_budget_id"])
for role, content in result["messages"]:
    print(f"  [{role}] {content}")

## Part 4: Spend tree — full ledger audit

After the run, inspect every authorization event in the fleet budget. The ledger shows which agent spent what, which events were confirmed, and which were denied — a complete audit trail.

In [ ]:
# Uses the fleet budget from Part 2
# (If you ran Part 3, replace `fleet.id` with `result["fleet_budget_id"]`)

page = client.get_ledger(fleet.id, page=0, size=50)

print(f"Fleet budget: {fleet.id}")
print(f"Total events: {page.total_elements}")
print()
print(f"  {'DECISION':<12}  {'AMOUNT':>8}  {'AGENT':<18}  DENIAL")
print(f"  {'─'*12}  {'─'*8}  {'─'*18}  {'─'*30}")

for ev in page.events:
    icon = {"CONFIRMED": "✓", "DENIED": "✗", "VOIDED": "∅", "AUTHORIZED": "○"}.get(ev.decision, "?")
    amount = ev.confirmed_quantity or ev.requested_quantity
    denial = ev.denial_reason or ""
    agent = (ev.agent_id or "")[:18]
    print(f"  {icon} {ev.decision:<11}  ${amount:>7.2f}  {agent:<18}  {denial}")

# Final fleet state
final = client.get_budget(fleet.id)
print(f"\nFleet total:  spent=${final.quantity_spent:.2f}  reserved=${final.quantity_reserved:.2f}  available=${final.available_quantity:.2f}")

## Part 5: `void_tree` — atomically cancel a graph run

When a LangGraph run fails mid-flight (exception, user cancellation, timeout), sub-agents may have live `AUTHORIZED` reservations that are holding fleet capacity. These stay frozen until `authorization_expiry_seconds` elapses unless you release them explicitly.

`void_tree` traverses the full causal chain rooted at a given event and voids every event in one call. All reserved capacity returns to the fleet immediately.

The pattern:
1. Before the run, authorize a root event with the full job budget (`max_subtree_quantity`)
2. Use `figuard_scope(root.event_id)` so every child authorization links to the root automatically
3. On failure, call `void_tree(root.event_id)` — releases everything in one round-trip

In [ ]:
from uuid import uuid4
from figuard import FiGuardClient, figuard_scope
from langchain_core.tools import ToolException
from figuard.integrations.langchain import FiGuardCallbackHandler

client = FiGuardClient()

# Fresh fleet for this example
fleet2 = client.create_budget(
    user_id="travel_orchestrator",
    total_limit=1000.00,
    currency="USD",
    expires_in="24h",
)

fleet2_token = fleet2.primary_token.session_token

flight_token2 = client.create_delegation_token(
    budget_id=fleet2.id,
    session_token=fleet2_token,
    label="flight-booker",
    caps=[{"category": "flights", "limit": 600.00}],
)
hotel_token2 = client.create_delegation_token(
    budget_id=fleet2.id,
    session_token=fleet2_token,
    label="hotel-booker",
    caps=[{"category": "hotels", "limit": 500.00}],
)

flight_handler2 = FiGuardCallbackHandler(
    client=client,
    session_token=flight_token2.session_token,
    agent_id="flight-booker",
    tool_category_map={"book_flight": "flights"},
)
hotel_handler2 = FiGuardCallbackHandler(
    client=client,
    session_token=hotel_token2.session_token,
    agent_id="hotel-booker",
    tool_category_map={"book_hotel": "hotels"},
)

# Step 1 — Orchestrator creates a root event for the whole job
# max_subtree_quantity declares the chain cap across ALL child events
root = client.authorize(
    session_token=fleet2.primary_token.session_token,
    agent_id="orchestrator",
    action_type="TRIP_PLAN_JOB",
    description="LA trip planning — root",
    requested_quantity=0.01,     # nominal $0.01 reserves the orchestrator's slot;
                                 # real cap is enforced via max_subtree_quantity
    idempotency_key=f"trip-{uuid4().hex[:8]}",
)
print(f"Root event: {root.event_id}  authorized={root.is_authorized}")

# Step 2 — Run sub-agents inside figuard_scope so their events link to root
# All handler.on_tool_start() calls inside the scope automatically set
# parent_event_id=root.event_id, building the causal tree.
with figuard_scope(root.event_id):
    run_id = uuid4()
    try:
        flight_handler2.on_tool_start(
            {"name": "book_flight"},
            '{"amount": 280.0, "description": "NYC to LAX"}',
            run_id=run_id,
        )
        flight_handler2.on_tool_end("NYC to LAX confirmed", run_id=run_id)
        print("  ✓  book_flight  $280.00  AUTHORIZED")
    except ToolException as e:
        print(f"  ✗  book_flight  DENIED: {e}")

    run_id2 = uuid4()
    try:
        hotel_handler2.on_tool_start(
            {"name": "book_hotel"},
            '{"amount": 420.0, "description": "Holiday Inn 3 nights"}',
            run_id=run_id2,
        )
        # Simulate: hotel tool raises an error mid-execution (payment gateway down)
        raise RuntimeError("Payment gateway timeout")
    except RuntimeError as e:
        hotel_handler2.on_tool_error(e, run_id=run_id2)  # releases the $420 reservation
        print(f"  ✗  book_hotel   $420.00  TOOL ERROR — reservation released")

# Step 3 — Simulate orchestrator deciding to cancel the whole run
# void_tree traverses the causal chain rooted at `root.event_id` and voids
# every event linked to it — the flight confirmation, the hotel failure, the root.
# All capacity returns to the fleet immediately.
void_result = client.void_tree(
    event_id=root.event_id,
    reason="TRIP_CANCELLED_BY_USER",
)
print(f"\nvoid_tree: released {void_result.voided_count} event(s), total ${void_result.total_quantity_released:.2f}")

# Fleet should be back to $1,000 available
fleet2_final = client.get_budget(fleet2.id)
print(f"Fleet available after void: ${fleet2_final.available_quantity:.2f}  (started at $1,000.00)")

Expected output:
```
Root event: evt_...  authorized=True
  ✓  book_flight  $280.00  AUTHORIZED
  ✗  book_hotel   $420.00  TOOL ERROR — reservation released

void_tree: released 2 event(s), total $280.01  # $280.00 flight + $0.01 root
Fleet available after void: $1000.00  (started at $1,000.00)
```

Without `void_tree`, the $280 flight reservation would stay frozen until the budget's `authorization_expiry_seconds` elapsed. With it, all capacity is returned in one round-trip — no waiting, no capacity leak.

## Summary — Fleet patterns for LangGraph

| Pattern | Code |
|---|---|
| Create fleet budget | `client.create_budget(total_limit=1000, ...)` |
| Issue delegation token | `client.create_delegation_token(budget_id, label, caps=[{"category": ..., "limit": ...}])` |
| Wire sub-agent handler | `FiGuardCallbackHandler(client, session_token=delegation_token.session_token, ...)` |
| Pass to graph node | `node.invoke(state, config={"callbacks": [handler]})` |
| Denial in tool result | `handle_tool_error=True` on inner `AgentExecutor` |
| Cancel run atomically | `client.void_tree(root_event_id, reason=...)` |
| Causal chain linking | `with figuard_scope(root_event_id): ...` |
| Fleet ledger | `client.get_ledger(fleet.id)` |

### Denial codes you'll encounter

| Code | Meaning |
|---|---|
| `DELEGATE_CAP_EXCEEDED` | Sub-agent hit its per-token cap |
| `BUDGET_EXHAUSTED` | Fleet total is depleted |
| `ALLOCATION_EXHAUSTED` | Category allocation (e.g. flights) is depleted |
| `SUBTREE_CAP_EXCEEDED` | Per-chain `max_subtree_quantity` exceeded |
| `BUDGET_EXPIRED` | Delegation token or fleet budget has expired |

With `handle_tool_error=True` on the inner `AgentExecutor`, the LLM receives the denial string as a tool result and can adapt: suggest a cheaper option, ask the user to increase the budget, or surface the limit as a final message.

## API reference

- [FiGuard GitHub](https://github.com/figuard/figuard-core)
- [Fleet agents guide](https://github.com/figuard/figuard-core/blob/main/docs/fleet-agents.md)
- [LangGraph integration guide](https://github.com/figuard/figuard-core/blob/main/docs/integrations/langgraph.md)
- [Denial handling](https://github.com/figuard/figuard-core/blob/main/docs/denial-handling.md)
- [Python SDK docs](https://github.com/figuard/figuard-core/tree/main/sdk/python)